# <B><U><I>mxmh_case2_model_building </U>----</B></I>

In [1]:

# MXMH CASE 2 - MODEL BUILDING NOTEBOOK
# Music Effects Prediction Model

# IGNORE WARNINGS --

import warnings
warnings.filterwarnings("ignore")

# IMPORTING ALL NECESSARY LIBRARIES --


import numpy as np
import pandas as pd

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

print("warning is ignored:📊📊")
print("ALL NECESSARY LIBRARIES ARE IMPORTED SUCESSFULLY:")
print("🎉🎉🎉🎉")

warning is ignored:📊📊
ALL NECESSARY LIBRARIES ARE IMPORTED SUCESSFULLY:
🎉🎉🎉🎉


In [2]:

# PATH SETUP --

CURRENT_DIR = Path.cwd()                         
NOTEBOOKS_DIR = CURRENT_DIR.parent             
BASE_DIR = CURRENT_DIR.parents[1]               

PREPROCESSED_DIR = NOTEBOOKS_DIR / "preprocessed"
MODELS_DIR = BASE_DIR / "saved_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PREPROCESSED_DIR / "mxmh_case2_model_ready.csv"

print("Current Dir      :", CURRENT_DIR)
print("Preprocessed Dir :", PREPROCESSED_DIR)
print("Models Dir       :", MODELS_DIR)
print("Dataset Path     :", DATA_PATH)

Current Dir      : c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\mxmh_survey_results
Preprocessed Dir : c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\preprocessed
Models Dir       : c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\saved_models
Dataset Path     : c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\preprocessed\mxmh_case2_model_ready.csv


In [3]:

# LOAD PREPROCESSED DATASET --

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)
df.head()

Dataset loaded successfully
Shape: (728, 67)


,age,hours_per_day,while_working,instrumentalist,composer,exploratory,foreign_languages,bpm,freq_classical,freq_country,...,bpm_band_low_bpm,bpm_band_medium_bpm,bpm_band_high_bpm,bpm_band_very_high_bpm,listening_band_low_listener,listening_band_moderate_listener,listening_band_heavy_listener,mental_health_band_low,mental_health_band_moderate,mental_health_band_high
0,18.0,4.0,0,0,0,0,1,132.0,0,0,...,False,False,True,False,False,True,False,False,False,True
1,61.0,2.5,1,0,1,1,1,84.0,2,0,...,True,False,False,False,False,True,False,False,False,True
2,18.0,4.0,1,0,0,1,0,107.0,0,0,...,False,True,False,False,False,True,False,False,False,True
3,18.0,5.0,1,1,1,1,1,86.0,1,2,...,True,False,False,False,False,True,False,False,False,True
4,18.0,3.0,1,1,0,1,1,66.0,2,0,...,True,False,False,False,False,True,False,False,True,False


In [4]:
print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values:", df.isnull().sum().sum())

Shape: (728, 67)

Columns:
 ['age', 'hours_per_day', 'while_working', 'instrumentalist', 'composer', 'exploratory', 'foreign_languages', 'bpm', 'freq_classical', 'freq_country', 'freq_edm', 'freq_folk', 'freq_gospel', 'freq_hip_hop', 'freq_jazz', 'freq_k_pop', 'freq_latin', 'freq_lofi', 'freq_metal', 'freq_pop', 'freq_rnb', 'freq_rap', 'freq_rock', 'freq_video_game_music', 'mental_symptom_score', 'mental_symptom_avg', 'music_engagement_score', 'genre_diversity_score', 'calm_music_score', 'high_energy_score', 'mainstream_music_score', 'music_effects_label', 'streaming_service_apple music', 'streaming_service_i do not use a streaming service.', 'streaming_service_other streaming service', 'streaming_service_pandora', 'streaming_service_spotify', 'streaming_service_youtube music', 'fav_genre_classical', 'fav_genre_country', 'fav_genre_edm', 'fav_genre_folk', 'fav_genre_gospel', 'fav_genre_hip hop', 'fav_genre_jazz', 'fav_genre_k pop', 'fav_genre_latin', 'fav_genre_lofi', 'fav_genre_metal'

In [5]:

# TARGET DISTRIBUTION --

print("music_effects_label distribution:\n")
print(df["music_effects_label"].value_counts().sort_index())

music_effects_label distribution:

music_effects_label
0    542
1    169
2     17
Name: count, dtype: int64


In [6]:

# DEFINE FEATURES AND TARGET --

target_col = "music_effects_label"

# drop target + original target one-hot columns if present
drop_cols = [target_col]

# safety: if original target text columns got one-hot encoded, remove them
target_like_cols = [col for col in df.columns if col.startswith("music_effects_")]
drop_cols.extend(target_like_cols)

# also if raw music_effects text column exists
if "music_effects" in df.columns:
    drop_cols.append("music_effects")

drop_cols = list(set(drop_cols))

X = df.drop(columns=drop_cols, errors="ignore")
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nDropped columns:", drop_cols)

X shape: (728, 63)
y shape: (728,)

Dropped columns: ['music_effects_label', 'music_effects_improve', 'music_effects_worsen', 'music_effects_no effect']


In [7]:

# TRAIN TEST SPLIT --

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (582, 63)
X_test : (146, 63)
y_train: (582,)
y_test : (146,)


# <B><U><I>MODEL 1 </U>----</B></I>

In [8]:

# LOGISTIC REGRESSION --

lr_model = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=42
)

lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)

lr_acc = accuracy_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds, average="weighted")

print("Logistic Regression Accuracy :", lr_acc)
print("Logistic Regression F1       :", lr_f1)

Logistic Regression Accuracy : 0.5342465753424658
Logistic Regression F1       : 0.5763686349509922


In [9]:
print("Classification Report — Logistic Regression\n")
print(classification_report(y_test, lr_preds))

Classification Report — Logistic Regression

              precision    recall  f1-score   support

           0       0.80      0.58      0.67       109
           1       0.25      0.41      0.31        34
           2       0.08      0.33      0.13         3

    accuracy                           0.53       146
   macro avg       0.38      0.44      0.37       146
weighted avg       0.66      0.53      0.58       146



# <b><u><i> MODEL 2</U> ----</B></I>

In [10]:

# RANDOM FOREST --

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_preds)
rf_f1 = f1_score(y_test, rf_preds, average="weighted")

print("Random Forest Accuracy :", rf_acc)
print("Random Forest F1       :", rf_f1)

Random Forest Accuracy : 0.7397260273972602
Random Forest F1       : 0.6348829683960737


In [11]:
print("Classification Report — Random Forest\n")
print(classification_report(y_test, rf_preds))

Classification Report — Random Forest

              precision    recall  f1-score   support

           0       0.74      0.99      0.85       109
           1       0.00      0.00      0.00        34
           2       0.00      0.00      0.00         3

    accuracy                           0.74       146
   macro avg       0.25      0.33      0.28       146
weighted avg       0.56      0.74      0.63       146



# <B><U><I>MODEL 3 </U>----</B></I>

In [12]:

# XGBOOST --

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_preds)
xgb_f1 = f1_score(y_test, xgb_preds, average="weighted")

print("XGBoost Accuracy :", xgb_acc)
print("XGBoost F1       :", xgb_f1)

XGBoost Accuracy : 0.7397260273972602
XGBoost F1       : 0.6662602524169298


In [13]:
print("Classification Report — XGBoost\n")
print(classification_report(y_test, xgb_preds))

Classification Report — XGBoost

              precision    recall  f1-score   support

           0       0.76      0.96      0.85       109
           1       0.43      0.09      0.15        34
           2       0.00      0.00      0.00         3

    accuracy                           0.74       146
   macro avg       0.39      0.35      0.33       146
weighted avg       0.66      0.74      0.67       146



In [14]:

# MODEL COMPARISON --

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        lr_acc,
        rf_acc,
        xgb_acc
    ],
    "Weighted_F1": [
        lr_f1,
        rf_f1,
        xgb_f1
    ]
})

results = results.sort_values(by=["Weighted_F1", "Accuracy"], ascending=False).reset_index(drop=True)

results

,Model,Accuracy,Weighted_F1
0,XGBoost,0.739726,0.666260
1,Random Forest,0.739726,0.634883
2,Logistic Regression,0.534247,0.576369


In [16]:
# -----------------------------
# SAVE BEST MODEL --

best_model = xgb_model
best_model_name = "xgboost"

print("Best model selected:", best_model_name)

Best model selected: xgboost


In [23]:
from pathlib import Path
import joblib
import json

# current notebook folder
CURRENT_DIR = Path.cwd()

# notebooks folder
NOTEBOOKS_DIR = CURRENT_DIR.parent

# notebooks/models folder
MODELS_DIR = NOTEBOOKS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Current dir :", CURRENT_DIR)
print("Notebooks dir:", NOTEBOOKS_DIR)
print("Models dir   :", MODELS_DIR)

Current dir : c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\mxmh_survey_results
Notebooks dir: c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks
Models dir   : c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\models


In [24]:
model_path = MODELS_DIR / "mxmh_case2_music_effects_xgboost.pkl"
joblib.dump(best_model, model_path)

print("Model saved successfully")
print("Saved at:", model_path)

Model saved successfully
Saved at: c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\models\mxmh_case2_music_effects_xgboost.pkl


In [26]:
feature_columns = X.columns.tolist()

features_path = MODELS_DIR / "mxmh_case2_feature_columns.json"
with open(features_path, "w") as f:
    json.dump(feature_columns, f)

print("Feature columns saved successfully")
print("Saved at:", features_path)

Feature columns saved successfully
Saved at: c:\Users\mohit\OneDrive\Desktop\Calmify AI\Calmify_AI\Calmify_AI\notebooks\models\mxmh_case2_feature_columns.json
